# 실습 3회: 재무제표 테이블 추출 및 데이터 정제

서울대학교 핀테크 실습 과정 | 12기

---

## 학습 목표

1. HTML 테이블의 구조(`table`, `tr`, `td`, `th`, `rowspan`, `colspan`)를 이해한다.
2. 감사보고서 내 5개 재무제표 테이블을 제목·내용 키워드로 식별한다.
3. `pandas.read_html()`을 활용하여 HTML 테이블을 `DataFrame`으로 변환한다.
4. 괄호 음수, 천단위 구분자 등 재무 수치 데이터를 정제한다.
5. 주석 섹션의 5단계 계층 구조를 스택 알고리즘으로 파싱하여 JSON 트리로 저장한다.
6. 재무제표와 주석을 연결하는 메타데이터 스키마를 CSV로 저장한다.

---

## 전체 처리 파이프라인

```
[공통] HTML 로드 + BeautifulSoup 파싱 (2회차 복습)
          |
    +-----+-----+
    |           |
[경로 A]     [경로 B]
재무제표       주석
    |           |
테이블 탐색   노드 수집
    |           |
DataFrame    계층 파싱
변환          (스택)
    |           |
수치 정제    JSON 저장
    |           |
CSV 저장    meta.csv
```

---

## Step 1. 패키지 설치

Google Colab을 포함한 모든 환경에서 동작하도록 필요한 라이브러리를 먼저 설치한다.

| 라이브러리 | 용도 |
|:---|:---|
| `beautifulsoup4` | HTML/XML 파싱의 핵심 라이브러리 |
| `lxml` | C 기반 고성능 HTML 파서 (BeautifulSoup 백엔드) |
| `html5lib` | HTML5 표준 준수 파서 (비정형 HTML 대응) |
| `pandas` | DataFrame 자료구조 및 `read_html()` 제공 |

`pandas` 는 Google Colab에 기본 설치되어 있으나, 로컬 환경에서는 별도 설치가 필요할 수 있다.

In [56]:
# Colab 및 로컬 환경 공통 패키지 설치
!pip install beautifulsoup4 lxml html5lib --quiet
print("패키지 설치 완료")

패키지 설치 완료


---

## Step 2. 파일 준비 및 기본 설정

2회차에서 사용한 파일 로드 방식을 그대로 재사용한다.
이 셀에서는 환경 감지, 파일 경로 설정, 유틸리티 함수 정의를 한 번에 수행한다.

### Google Colab에서 파일 업로드

2회차와 동일하게 두 가지 방법 중 하나를 선택한다.

**방법 A: 직접 업로드**
```python
from google.colab import files
uploaded = files.upload()   # 파일 선택 창이 열림
```

**방법 B: Google Drive 연동 (권장)**
```python
from google.colab import drive
drive.mount('/content/drive')
# 이후 경로: '/content/drive/MyDrive/실습데이터셋/감사보고서_2024.htm'
```

In [57]:
import re
import json
import csv
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd
from bs4 import BeautifulSoup, Tag, NavigableString

# 실행 환경 감지
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"실행 환경: {'Google Colab' if IN_COLAB else '로컬 Jupyter'}")

실행 환경: 로컬 Jupyter


In [58]:
# 파일 경로 설정 (2회차와 동일한 방법)
if IN_COLAB:
    # 방법 A: 직접 업로드
    # from google.colab import files
    # uploaded = files.upload()
    # HTML_PATH = Path(list(uploaded.keys())[0])

    # 방법 B: Google Drive
    # from google.colab import drive
    # drive.mount('/content/drive')
    # HTML_PATH = Path('/content/drive/MyDrive/실습데이터셋/감사보고서_2024.htm')

    HTML_PATH  = Path("/content/감사보고서_2024.htm")
    OUTPUT_DIR = Path("/content/output")
else:
    HTML_PATH  = Path("../실습데이터셋/감사보고서_2024.htm")
    OUTPUT_DIR = Path("../output")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
YEAR = 2024

print(f"HTML 파일: {HTML_PATH}  (존재: {HTML_PATH.exists()})")
print(f"출력 디렉토리: {OUTPUT_DIR}")

HTML 파일: ../실습데이터셋/감사보고서_2024.htm  (존재: True)
출력 디렉토리: ../output


In [59]:
# 2회차 유틸리티 함수 재사용

def load_html(path: Path) -> str:
    for enc in ("euc-kr", "cp949", "utf-8"):
        try:
            return path.read_text(encoding=enc)
        except (UnicodeDecodeError, LookupError):
            continue
    return path.read_bytes().decode("utf-8", errors="replace")


def get_soup(html_text: str) -> BeautifulSoup:
    for parser in ("lxml", "html5lib", "html.parser"):
        try:
            return BeautifulSoup(html_text, parser)
        except Exception:
            continue
    return BeautifulSoup(html_text, "html.parser")


def get_section_level(tag: Tag) -> Optional[int]:
    for cls in (tag.get("class") or []):
        m = re.match(r"(?i)^section-(\d+)$", str(cls).strip())
        if m:
            return int(m.group(1))
    return None


def is_section_header(tag: Tag) -> bool:
    return isinstance(tag, Tag) and tag.name in ("h2", "h3") and get_section_level(tag) in (1, 2)


html_text = load_html(HTML_PATH)
soup      = get_soup(html_text)
print(f"로드 완료 | 총 테이블 수: {len(soup.find_all('table')):,}")

로드 완료 | 총 테이블 수: 278


---

## Step 3. HTML 테이블 구조 이해

### 3.1 HTML 테이블의 기본 구조

재무제표는 HTML의 `<table>` 태그 안에 구현된다.
`<table>`은 여러 태그의 중첩으로 구성되는 복잡한 구조이다.

```
<table>          ← 표 전체를 감싸는 컨테이너
  <tr>           ← 한 행(Table Row)
    <th>과목</th> ← 헤더 셀(Table Header): 굵게 표시
    <th>당기</th>
    <th>전기</th>
  </tr>
  <tr>           ← 다음 행
    <td>현금</td> ← 데이터 셀(Table Data)
    <td>50,000</td>
    <td>45,000</td>
  </tr>
</table>
```

| 태그 | 이름 | 역할 |
|:---|:---|:---|
| `<table>` | Table | 표 전체 컨테이너 |
| `<tr>` | Table Row | 하나의 행 |
| `<th>` | Table Header | 헤더 셀 (굵은 글씨로 표시) |
| `<td>` | Table Data | 일반 데이터 셀 |

---

### 3.2 셀 병합: rowspan과 colspan

재무제표 헤더에는 여러 셀을 하나로 합치는 **병합** 이 자주 사용된다.

- **`colspan`**: 가로(열) 방향으로 셀을 병합한다.
  - `<th colspan="2">당기</th>` → "당기" 셀이 가로 2칸을 차지한다.
- **`rowspan`**: 세로(행) 방향으로 셀을 병합한다.
  - `<th rowspan="2">과목</th>` → "과목" 셀이 세로 2행을 차지한다.

```html
<table>
  <tr>
    <th rowspan="2">과목</th>      ← 세로 2칸 차지
    <th colspan="2">당기</th>      ← 가로 2칸 차지 (금액 + 비고)
    <th colspan="2">전기</th>      ← 가로 2칸 차지
  </tr>
  <tr>
    <!-- 과목 셀은 rowspan으로 이미 차지했으므로 없음 -->
    <th>금액</th><th>비고</th>     ← 당기의 하위 헤더
    <th>금액</th><th>비고</th>     ← 전기의 하위 헤더
  </tr>
  <tr>
    <td>현금</td>
    <td>50,000</td><td>-</td>
    <td>45,000</td><td>-</td>
  </tr>
</table>
```

**주의**: 셀 병합이 있을 경우 BeautifulSoup으로 행을 직접 파싱하면
셀 수가 행마다 달라져 데이터 정렬이 맞지 않는 문제가 생긴다.
`pandas.read_html()`은 이를 내부적으로 자동 처리한다.

---

### 3.3 pandas.read_html()이란

`pd.read_html(html_string)` 함수는 HTML 텍스트 안의 모든 `<table>` 을 찾아
`pandas DataFrame` 목록으로 변환해 준다.

```python
import pandas as pd

dfs = pd.read_html("<table>...</table>")  # 반환: DataFrame 리스트
df  = dfs[0]                              # 첫 번째 테이블
```

`colspan`/`rowspan`이 있으면 **멀티인덱스(MultiIndex) 컬럼**이 생성된다.
멀티인덱스란 컬럼 이름이 여러 레벨의 계층 구조를 가지는 것을 의미한다.

In [60]:
# [실습 3-1] rowspan/colspan을 포함한 재무제표 헤더 구조 실습

sample_table = """
<table>
  <tr>
    <th rowspan="2">과목</th>
    <th rowspan="2">주석</th>
    <th colspan="2">당기 (2024.12.31)</th>
    <th colspan="2">전기 (2023.12.31)</th>
  </tr>
  <tr>
    <th>금액</th><th>비고</th>
    <th>금액</th><th>비고</th>
  </tr>
  <tr>
    <td>현금및현금성자산</td><td>4</td>
    <td>50,000</td><td>-</td><td>45,000</td><td>-</td>
  </tr>
  <tr>
    <td>매출채권</td><td>5</td>
    <td>(120,000)</td><td>-</td><td>110,000</td><td>-</td>
  </tr>
</table>
"""

tbl_soup = BeautifulSoup(sample_table, "html.parser").find("table")

print("[BeautifulSoup: 행별 셀 출력]")
for i, tr in enumerate(tbl_soup.find_all("tr"), 1):
    cells = []
    for cell in tr.find_all(["td", "th"]):
        text = cell.get_text(strip=True)
        rs   = cell.get("rowspan", "")
        cs   = cell.get("colspan", "")
        annotation = f"(rs={rs})" if rs else (f"(cs={cs})" if cs else "")
        cells.append(f"{text}{annotation}")
    print(f"  행 {i}: {cells}")

[BeautifulSoup: 행별 셀 출력]
  행 1: ['과목(rs=2)', '주석(rs=2)', '당기 (2024.12.31)(cs=2)', '전기 (2023.12.31)(cs=2)']
  행 2: ['금액', '비고', '금액', '비고']
  행 3: ['현금및현금성자산', '4', '50,000', '-', '45,000', '-']
  행 4: ['매출채권', '5', '(120,000)', '-', '110,000', '-']


In [61]:
# [실습 3-2] pandas.read_html()로 colspan/rowspan 자동 처리

dfs = pd.read_html(sample_table, flavor="lxml")
print(f"read_html() 결과: {len(dfs)}개 DataFrame")
print()
print(dfs[0].to_string())
print()
print(f"컬럼 타입: {dfs[0].columns.tolist()}")
print("(멀티인덱스 컬럼이 생성됨에 주목)")

read_html() 결과: 1개 DataFrame

         과목 주석 당기 (2024.12.31)    전기 (2023.12.31)   
         과목 주석              금액 비고              금액 비고
0  현금및현금성자산  4           50000  -           45000  -
1      매출채권  5       (120,000)  -          110000  -

컬럼 타입: [('과목', '과목'), ('주석', '주석'), ('당기 (2024.12.31)', '금액'), ('당기 (2024.12.31)', '비고'), ('전기 (2023.12.31)', '금액'), ('전기 (2023.12.31)', '비고')]
(멀티인덱스 컬럼이 생성됨에 주목)


/var/folders/cv/gqggtkw96sx21g79rcxw1rph0000gn/T/ipykernel_25136/3657257680.py:3: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(sample_table, flavor="lxml")


---

## Step 4. 재무제표 탐색 전략: 제목 키워드 매칭

### 4.1 DART HTML의 재무제표 구조: 제목 테이블 + 데이터 테이블 쌍

DART 감사보고서에서 각 재무제표는 항상 **두 개의 테이블이 연속으로** 배치된다.

```
[제목 테이블] (소형)
  <table>
    <td>손 익 계 산 서</td>           ← 제목 (글자 사이에 공백 있음!)
    <td>제 56 기 : 2024년 1월 1일부터 2024년 12월 31일까지</td>
    <td>삼성전자주식회사</td>
    <td>(단위 : 백만원)</td>
  </table>

[데이터 테이블] (대형, 바로 다음 형제)
  <table>
    <tr><td>Ⅰ. 매 출 액</td><td>29</td><td>209,052,241</td>...</tr>
    <tr><td>Ⅱ. 매 출 원 가</td>...</tr>
    ...
  </table>
```

두 가지 함정:
1. **글자 사이 공백**: 제목이 `"손익계산서"`가 아닌 `"손 익 계 산 서"`로 저장되어 있어
   단순 문자열 검색으로는 매칭되지 않는다.
2. **제목 텍스트가 본문에도 등장**: 감사 의견문, 주석 등에서도 "손익계산서"가 언급된다.
   이 텍스트를 시작점으로 삼으면 엉뚱한 표(종속기업 재무정보 등)를 가져오게 된다.

**올바른 전략**: 제목 텍스트가 `<td>` 셀 내부에 있는 경우만 시작점으로 인식하고,
그 테이블의 바로 다음 형제 테이블을 데이터 테이블로 선택한다.

---

### 4.2 문제의 어려움: 278개의 표 중 5개를 골라야 한다

감사보고서 HTML에는 약 278개의 `<table>` 태그가 존재한다.
감사 의견문, 주석, 참고 자료 등 모든 표가 포함되어 있기 때문이다.
단순히 "모든 `<table>` 을 가져오면 되지 않느냐"는 접근은 통하지 않는다.

우리가 원하는 것은 딱 5개: 재무상태표, 손익계산서, 포괄손익계산서, 자본변동표, 현금흐름표.

---

### 4.2 2단계 탐색 전략

```
[1단계] 제목 텍스트 탐색
    soup.find_all(string=re.compile("재무상태표"))
    → "재무상태표" 텍스트를 포함한 모든 노드를 찾는다
         ↓
[2단계] 키워드 검증
    제목 근처의 <table>이 실제 해당 재표인지
    "자산", "부채", "자본" 등 고유 계정과목 키워드로 확인한다
```

---

### 4.3 텍스트 노드 탐색: soup.find_all(string=...)

BeautifulSoup의 `find_all(string=...)` 은 **텍스트 노드(NavigableString)** 를 검색한다.
태그가 아닌 실제 문자열 내용을 기준으로 찾는다는 점에서 `find_all("p")` 와 다르다.

```python
# "재무상태표"라는 텍스트를 포함한 모든 텍스트 노드 검색
nodes = soup.find_all(string=re.compile("재무상태표"))

for node in nodes:
    parent = node.parent   # 이 텍스트를 담고 있는 태그
    print(parent.name, parent.get("class"))
```

---

### 4.4 가까운 테이블로 이동하기

제목 텍스트 노드를 찾은 다음, 그 **근처(형제 또는 조상 방향)**에 있는 `<table>` 을 탐색한다.
DOM 트리에서 제목과 표는 형제 관계이거나, 공통 부모 아래에 있는 경우가 대부분이다.

```
<div>
  <p>"재무상태표"  ← 텍스트 노드 발견
  <table>...      ← 찾고자 하는 표 (형제 노드)
</div>
```

만약 형제에서 못 찾으면 **부모(parent)로 올라가며** 다시 형제를 탐색한다.
이를 반복하면서 키워드 검증을 통과한 첫 번째 `<table>` 을 반환한다.

In [62]:
# [실습 4-1] 5개 재무제표 정의
#
# DART HTML 특이사항:
#   각 재무제표 제목이 <table><td>손 익 계 산 서</td></table> 형태의
#   '제목 테이블' 셀 안에 글자 사이 공백을 넣어 표시된다.
#   따라서 spaced 버전(예: "손 익 계 산 서")을 우선 탐색 목록 앞에 배치한다.

STATEMENT_TITLES: Dict[str, List[str]] = {
    "재무상태표"    : ["재 무 상 태 표", "재무상태표", "대차대조표"],
    "손익계산서"    : ["손 익 계 산 서", "손익계산서"],
    "포괄손익계산서": ["포 괄 손 익 계 산 서", "포괄손익계산서"],
    "자본변동표"    : ["자 본 변 동 표", "자본변동표"],
    "현금흐름표"    : ["현 금 흐 름 표", "현금흐름표"],
}

STATEMENT_KEYWORDS: Dict[str, List[str]] = {
    "재무상태표"    : ["자산", "부채", "자본", "유동자산", "비유동자산"],
    "손익계산서"    : ["매출액", "영업이익", "당기순이익", "매출원가"],
    "포괄손익계산서": ["기타포괄손익", "포괄손익총계", "당기순이익"],
    "자본변동표"    : ["자본금", "이익잉여금"],
    "현금흐름표"    : ["영업활동", "투자활동", "재무활동", "현금및현금성자산"],
}

STATEMENT_CODES: Dict[str, str] = {
    "재무상태표"    : "BS",
    "손익계산서"    : "IS",
    "포괄손익계산서": "OCI",
    "자본변동표"    : "CSE",
    "현금흐름표"    : "CF",
}

print("5개 재무제표 정의:")
for name, code in STATEMENT_CODES.items():
    kws = STATEMENT_KEYWORDS[name]
    print(f"  {code} ({name}): 검증 키워드 = {kws}")

5개 재무제표 정의:
  BS (재무상태표): 검증 키워드 = ['자산', '부채', '자본', '유동자산', '비유동자산']
  IS (손익계산서): 검증 키워드 = ['매출액', '영업이익', '당기순이익', '매출원가']
  OCI (포괄손익계산서): 검증 키워드 = ['기타포괄손익', '포괄손익총계', '당기순이익']
  CSE (자본변동표): 검증 키워드 = ['자본금', '이익잉여금']
  CF (현금흐름표): 검증 키워드 = ['영업활동', '투자활동', '재무활동', '현금및현금성자산']


In [63]:
# [실습 4-2] 제목 텍스트를 포함하는 노드 탐색
# soup.find_all(string=...) 으로 텍스트 노드 검색

target_title = "재무상태표"
title_nodes = soup.find_all(string=re.compile(re.escape(target_title), re.I))

print(f"'{target_title}' 텍스트 노드 수: {len(title_nodes)}")
print()
for node in title_nodes[:5]:
    parent = node.parent
    print(f"  텍스트: {str(node).strip()[:50]!r}")
    print(f"  부모 태그: <{parent.name}> class={parent.get('class')}")
    print()

'재무상태표' 텍스트 노드 수: 8

  텍스트: '우리는 별첨된 삼성전자주식회사(이하 "회사")의 재무제표를 감사하였습니다. 해당 재무제표는'
  부모 태그: <p> class=None

  텍스트: '회사는 계약상 내용의 실질과 금융부채의 정의에 따라 금융부채를 당기손익인식금융부채와 기타금'
  부모 태그: <p> class=None

  텍스트: '회사는 확정급여제도와 확정기여제도를 포함하는 다양한 형태의 퇴직연금제도를 운영하고 있습니다'
  부모 태그: <p> class=None

  텍스트: '회사는 리스개시일에 기초자산을 사용할 권리를 나타내는 사용권자산(리스자산)과 리스료를 지급'
  부모 태그: <p> class=None

  텍스트: '회사는 당기 및 전기 중 은행과의 매출채권 팩토링 계약을 통해 매출채권을 할인하였습니다. '
  부모 태그: <p> class=None



In [64]:
# [실습 4-3] 키워드 검증 함수

MIN_KEYWORD_MATCH = 2

def is_valid_financial_table(table_tag: Tag, statement_type: str) -> bool:
    """
    테이블이 해당 재무제표 유형인지 키워드로 검증한다.
    - 최소 MIN_KEYWORD_MATCH개 이상의 키워드가 있어야 함
    - 4자리 이상의 숫자(쉼표 포함)가 있어야 실제 데이터 테이블로 판단

    DART 특이사항:
      재무제표 계정과목이 '매 &nbsp; &nbsp; 출 &nbsp; &nbsp; 액' 형태로 작성됨.
      BeautifulSoup이 이를 '매 \\xa0 \\xa0 출 \\xa0 \\xa0 액'으로 변환하여
      단순 문자열 비교로는 '매출액' 키워드가 매칭되지 않는다.
      따라서 공백·nbsp를 모두 제거한 정규화 텍스트로 키워드를 비교한다.
    """
    raw_text  = table_tag.get_text()
    # 모든 공백류 문자(스페이스, \xa0, \t, \n 등) 제거 후 비교
    norm_text = re.sub(r'[\s\xa0]+', '', raw_text)
    keywords  = STATEMENT_KEYWORDS.get(statement_type, [])
    matched   = sum(1 for kw in keywords if kw.replace(' ', '') in norm_text)

    if matched < MIN_KEYWORD_MATCH:
        return False
    return bool(re.search(r'[\d,]{4,}', raw_text))


# 테스트: 모든 table 태그 중 재무상태표로 검증되는 것 찾기
all_tables = soup.find_all("table")
valid_bs_tables = [
    (i, tbl) for i, tbl in enumerate(all_tables)
    if is_valid_financial_table(tbl, "재무상태표")
]

print(f"전체 테이블 수: {len(all_tables)}")
print(f"재무상태표로 검증된 테이블: {len(valid_bs_tables)}개")
for idx, tbl in valid_bs_tables[:3]:
    row_count = len(tbl.find_all("tr"))
    print(f"  인덱스 {idx}: {row_count}행")

전체 테이블 수: 278
재무상태표로 검증된 테이블: 31개
  인덱스 11: 48행
  인덱스 17: 8행
  인덱스 20: 18행


In [65]:
# [실습 4-4] 제목 → 데이터 테이블 탐색 함수 (DART 구조 대응)
#
# DART 재무제표 HTML 패턴:
#   [제목 테이블] <table>
#                  <td>손 익 계 산 서</td>   ← 제목이 TD 셀 안에 있음
#                  <td>제 56 기 ...</td>
#                  <td>(단위: 백만원)</td>
#                </table>
#   [데이터 테이블] <table>                  ← 바로 다음 형제 테이블
#                    <tr><td>Ⅰ. 매출액</td><td>209,052,241</td></tr>
#                    ...
#                  </table>

def find_financial_table(soup_obj: BeautifulSoup, statement_type: str) -> Optional[Tag]:
    """
    DART 감사보고서의 '제목 테이블 → 데이터 테이블' 쌍 구조를 이용하여
    정확한 재무제표 데이터 테이블을 탐색한다.

    탐색 전략 (2단계):
      [1단계] 제목 텍스트가 <td>/<th> 셀 내부에 있는 테이블을 제목 테이블로 인식하고,
              그 다음 형제 <table>을 데이터 테이블 후보로 선정한다.
              → 본문 <p> 태그에서 언급되는 "손익계산서" 등은 무시된다.
      [2단계] 1단계 실패 시, 기존 방식(제목 텍스트 → 형제/조상 방향 탐색)으로 폴백.
    """
    titles = STATEMENT_TITLES.get(statement_type, [])

    # ── 1단계: TD/TH 셀 내부 제목 텍스트 → 제목 테이블 → 다음 형제 데이터 테이블 ──
    for title in titles:
        pattern = re.compile(re.escape(title), re.I)
        for node in soup_obj.find_all(string=pattern):
            cell = node.parent
            # <td> 또는 <th> 셀 내부 텍스트만 처리 (p/span 등의 본문 텍스트 제외)
            if not isinstance(cell, Tag) or cell.name not in ("td", "th"):
                continue

            title_table = cell.find_parent("table")
            if title_table is None:
                continue

            # 제목 테이블 이후 형제 노드를 순회하여 첫 번째 <table>을 후보로 선정
            for sib in title_table.next_siblings:
                if not isinstance(sib, Tag):
                    continue
                if sib.name != "table":
                    continue

                # 후보 테이블이 키워드 검증 통과 → 반환
                if is_valid_financial_table(sib, statement_type):
                    return sib

                # 키워드 미통과(빈 구분 테이블일 수 있음) → 한 번 더 다음 테이블 시도
                for sib2 in sib.next_siblings:
                    if isinstance(sib2, Tag) and sib2.name == "table":
                        if is_valid_financial_table(sib2, statement_type):
                            return sib2
                        break
                break  # 두 번째 시도도 실패하면 이 제목 노드 포기

    # ── 2단계 폴백: 제목 텍스트 → 형제/조상 방향 탐색 ──────────────────────────
    for title in titles:
        for node in soup_obj.find_all(string=re.compile(re.escape(title), re.I)):
            parent = node.parent
            depth  = 0
            while parent and parent.name != "body" and depth < 6:
                for sibling in parent.next_siblings:
                    if not isinstance(sibling, Tag):
                        continue
                    if sibling.name == "table":
                        if is_valid_financial_table(sibling, statement_type):
                            return sibling
                    for inner in sibling.find_all("table"):
                        if is_valid_financial_table(inner, statement_type):
                            return inner
                parent = parent.parent
                depth += 1

    return None


# ── 5개 재무제표 탐색 실행 ────────────────────────────────────────────────
found_tables: Dict[str, Optional[Tag]] = {}

print(f"{'재무제표':<15} {'코드':<6} {'결과':<12} {'행 수':<8} {'첫 번째 계정과목'}")
print("-" * 65)
for stmt in STATEMENT_CODES:
    tbl  = find_financial_table(soup, stmt)
    found_tables[stmt] = tbl
    code = STATEMENT_CODES[stmt]
    if tbl:
        rows      = len(tbl.find_all("tr"))
        first_tds = tbl.find_all("td")
        first_acc = ""
        for td in first_tds:
            t = td.get_text(" ", strip=True).replace("\xa0", "")
            if t and not re.match(r'^[\d,.()\-\s]+$', t):
                first_acc = t[:20]
                break
        print(f"{stmt:<15} {code:<6} {'발견':<12} {rows:<8} {first_acc}")
    else:
        print(f"{stmt:<15} {code:<6} {'미발견'}")

재무제표            코드     결과           행 수      첫 번째 계정과목
-----------------------------------------------------------------
재무상태표           BS     발견           48       자     산
손익계산서           IS     발견           16       Ⅰ. 매   출   액
포괄손익계산서         OCI    발견           8        Ⅰ. 당기순이익
자본변동표           CSE    발견           18       2023.1.1(전기초)
현금흐름표           CF     발견           32       Ⅰ. 영업활동 현금흐름


---

## Step 5. HTML 테이블 → pandas DataFrame 변환

### 5.1 pandas DataFrame이란

`DataFrame`은 pandas 라이브러리의 핵심 자료구조로,
**행(row)과 열(column)로 구성된 2차원 표**이다.

```python
import pandas as pd

df = pd.DataFrame({
    "과목":   ["현금", "매출채권"],
    "당기":   [50000,  120000],
    "전기":   [45000,  110000],
})

df.shape        # (2, 3) - 2행 3열
df.columns      # ['과목', '당기', '전기']
df["당기"]      # 당기 열 전체를 Series로 반환
df.iloc[0]      # 첫 번째 행 전체
```

엑셀 스프레드시트와 유사한 개념이며, 금융 데이터 분석에 가장 많이 쓰이는 자료구조이다.

---

### 5.2 HTML 테이블을 DataFrame으로 변환하는 두 가지 방법

**방법 1: `pd.read_html()` (권장)**

HTML 문자열을 받아 내부의 모든 `<table>` 을 자동으로 DataFrame 리스트로 변환한다.
`colspan`/`rowspan` 병합도 자동 처리하며, 멀티인덱스 컬럼을 생성한다.

```python
dfs = pd.read_html(str(table_tag), flavor="lxml")
df  = dfs[0]
```

장점: 빠르고 병합 처리가 자동화됨.
단점: 일부 비표준 HTML에서 오류가 발생할 수 있음.

**방법 2: 수동 파싱 (fallback)**

`BeautifulSoup`으로 `<tr>` → `<td>/<th>` 를 직접 순회하여 셀 텍스트를 추출한다.
`colspan` 만 처리하고 `rowspan` 은 무시하므로 완벽하지 않지만,
`read_html()` 이 실패했을 때 안정적인 대안이 된다.

---

### 5.3 폴백(Fallback) 설계 패턴

소프트웨어에서 "폴백(fallback)"이란 **우선 방법이 실패했을 때 자동으로 대안을 시도**하는 설계 패턴이다.
여기서는 `read_html()` → 실패 → `manual_parse_table()` 순으로 시도한다.

```python
def table_tag_to_df(table_tag):
    try:
        dfs = pd.read_html(str(table_tag), flavor="lxml")
        if dfs:
            return max(dfs, key=lambda d: d.shape[0] * d.shape[1])
    except Exception:
        pass                           # 실패 시 수동 파싱으로 진행

    return manual_parse_table(table_tag)  # 폴백
```

이 패턴은 실전 데이터 파이프라인에서 매우 중요하다.
모든 HTML이 표준을 완벽히 준수하지 않기 때문이다.

In [66]:
# [실습 5-1] read_html()로 재무상태표 변환 시도

bs_tag = found_tables.get("재무상태표")

if bs_tag:
    try:
        dfs = pd.read_html(str(bs_tag), flavor="lxml")
        df_raw = max(dfs, key=lambda d: d.shape[0] * d.shape[1]) if dfs else None
        print(f"read_html() 성공: {df_raw.shape}")
        print(f"컬럼: {df_raw.columns.tolist()[:6]}...")
        print()
        print(df_raw.head(8).to_string())
    except Exception as e:
        print(f"read_html() 실패: {e}")
else:
    print("재무상태표 테이블을 찾지 못했습니다.")

read_html() 성공: (47, 6)
컬럼: ['과 목', '주석', '제 56 (당) 기', '제 56 (당) 기.1', '제 55 (전) 기', '제 55 (전) 기.1']...

           과 목           주석 제 56 (당) 기 제 56 (당) 기.1 제 55 (전) 기 제 55 (전) 기.1
0          자 산          NaN        NaN          NaN        NaN          NaN
1   Ⅰ. 유 동 자 산          NaN        NaN     82320322        NaN     68548442
2  1. 현금및현금성자산        4, 28    1653766          NaN    6061451          NaN
3    2. 단기금융상품        4, 28   10187991          NaN      50071          NaN
4      3. 매출채권  4, 5, 7, 28   33840357          NaN   27363016          NaN
5       4. 미수금     4, 7, 28    3249731          NaN    1910054          NaN
6      5. 선급비용          NaN    1381781          NaN    1349755          NaN
7      6. 재고자산            8   29154115          NaN   29338151          NaN


/var/folders/cv/gqggtkw96sx21g79rcxw1rph0000gn/T/ipykernel_25136/2089878798.py:7: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(bs_tag), flavor="lxml")


In [67]:
# [실습 5-2] 수동 파싱 함수 (read_html 실패 시 대체)

def manual_parse_table(table_tag: Tag) -> Optional[pd.DataFrame]:
    """
    HTML 테이블 태그를 행/셀 단위로 직접 파싱한다.
    colspan은 동일 텍스트를 반복 삽입하여 처리한다.
    rowspan은 처리하지 않는다 (셀 수 불일치 가능).
    """
    rows: List[List[str]] = []
    for tr in table_tag.find_all("tr"):
        row: List[str] = []
        for cell in tr.find_all(["td", "th"]):
            text    = cell.get_text(" ", strip=True).replace("\xa0", " ")
            colspan = int(cell.get("colspan", 1))
            row.extend([text] * colspan)
        if row:
            rows.append(row)

    if not rows:
        return None

    max_cols = max(len(r) for r in rows)
    rows = [r + [""] * (max_cols - len(r)) for r in rows]

    try:
        return pd.DataFrame(rows[1:], columns=rows[0])
    except Exception:
        return pd.DataFrame(rows)


# 수동 파싱 결과 확인
if bs_tag:
    df_manual = manual_parse_table(bs_tag)
    if df_manual is not None:
        print(f"수동 파싱 결과: {df_manual.shape}")
        print(df_manual.head(6).to_string())

수동 파싱 결과: (47, 6)
  과                        목           주석  제 56 (당) 기  제 56 (당) 기  제 55 (전) 기  제 55 (전) 기
0               자          산                                                             
1              Ⅰ. 유  동  자  산                           82,320,322              68,548,442
2                1. 현금및현금성자산        4, 28   1,653,766               6,061,451            
3                  2. 단기금융상품        4, 28  10,187,991                  50,071            
4                    3. 매출채권  4, 5, 7, 28  33,840,357              27,363,016            
5                     4. 미수금     4, 7, 28   3,249,731               1,910,054            


In [68]:
# [실습 5-3] 통합 변환 함수: read_html 우선, 실패 시 수동 파싱

def table_tag_to_df(table_tag: Tag) -> Optional[pd.DataFrame]:
    """
    HTML 테이블 태그를 DataFrame으로 변환한다.
    read_html()을 우선 시도하고, 실패하면 수동 파싱을 사용한다.
    """
    if table_tag is None:
        return None

    try:
        dfs = pd.read_html(str(table_tag), flavor="lxml")
        if dfs:
            return max(dfs, key=lambda d: d.shape[0] * d.shape[1])
    except Exception:
        pass

    return manual_parse_table(table_tag)


# 5개 재무제표 모두 변환 시도
raw_dfs: Dict[str, Optional[pd.DataFrame]] = {}
print(f"{'재무제표':<15} {'형태'}")
print("-" * 35)
for stmt, tbl in found_tables.items():
    df = table_tag_to_df(tbl)
    raw_dfs[stmt] = df
    shape = str(df.shape) if df is not None else "변환 실패"
    print(f"{stmt:<15} {shape}")

재무제표            형태
-----------------------------------
재무상태표           (47, 6)
손익계산서           (15, 6)
포괄손익계산서         (7, 6)
자본변동표           (17, 7)
현금흐름표           (31, 6)


/var/folders/cv/gqggtkw96sx21g79rcxw1rph0000gn/T/ipykernel_25136/2237354713.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(table_tag), flavor="lxml")
/var/folders/cv/gqggtkw96sx21g79rcxw1rph0000gn/T/ipykernel_25136/2237354713.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(table_tag), flavor="lxml")
/var/folders/cv/gqggtkw96sx21g79rcxw1rph0000gn/T/ipykernel_25136/2237354713.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(table_tag), flavor="lxml")
/var/folders/cv/gqggtkw96sx21g79rcxw1rph0000gn/T/ipykernel_25136/2237354713.py:12: Futur

---

## Step 6. 수치 데이터 정제

### 6.1 왜 정제가 필요한가

HTML 테이블에서 추출한 셀 값은 모두 **문자열(str)** 이다.
`"50,000"` 은 숫자처럼 보이지만 파이썬은 이를 문자열로 인식한다.
따라서 덧셈, 비율 계산 등 수치 연산을 하려면 **숫자(float)로 변환**해야 한다.

문제는 감사보고서의 숫자 표기 방식이 일반적인 숫자와 다르다는 점이다.

---

### 6.2 DART 감사보고서의 수치 표현 규칙

감사보고서는 한국 회계 기준(K-IFRS)에 따라 다음과 같은 표기 관행을 따른다.

| 원시 텍스트 | 의미 | 변환 결과 |
|:---|:---|:---|
| `1,234,567` | 양수. 쉼표(`,`)는 천단위 구분자 (숫자 아님) | `1234567.0` |
| `(1,234,567)` | **음수**. 괄호는 마이너스를 의미 | `-1234567.0` |
| `-` 또는 `–` | 해당 없음 또는 0 | `0.0` |
| `` (빈 문자열) | 결측값 (해당 기간 없음) | `None` |
| `자산총계` | 계정과목명 텍스트 → 수치 아님 | `None` |

**괄호 음수 표현**은 한국 회계 관행에서 매우 흔하다.
`float("(1,234)")` 는 파이썬에서 오류를 발생시키므로 수동으로 처리해야 한다.

---

### 6.3 정규표현식으로 숫자 검증하기

`re.search(r'[\d,]{4,}', text)` 는 "4자리 이상의 숫자·쉼표 조합"을 탐색한다.
이를 통해 실제 데이터가 있는 테이블과 빈 테이블을 구분할 수 있다.

```
r'[\d,]{4,}'   →  [ ]  문자 클래스 (대괄호 안의 문자 중 하나)
                 \d     0-9 숫자
                  ,     쉼표
                {4,}   4회 이상 반복
```

즉, `"1,234"`, `"50,000"`, `"1,000,000"` 등을 탐지하는 패턴이다.

In [69]:
# [실습 6-1] 수치 변환 함수 구현 및 테스트

def parse_numeric(text: Any) -> Optional[float]:
    """
    재무제표 수치 텍스트를 float로 변환한다.
    괄호는 음수, 쉼표는 천단위 구분자, 대시는 0으로 처리한다.
    숫자가 없는 텍스트는 None을 반환한다.
    """
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return None

    s = str(text).strip().replace("\xa0", "").replace(" ", "")

    if s in ("-", "–", "—", ""):
        return 0.0

    is_negative = s.startswith("(") and s.endswith(")")
    if is_negative:
        s = s[1:-1]

    s = s.replace(",", "")

    try:
        value = float(s)
        return -value if is_negative else value
    except ValueError:
        return None


# 단위 추출 함수
def extract_unit_label(text: str) -> Optional[str]:
    """'(단위: 백만원)' 또는 '단위 : 천원' 형태에서 단위를 추출한다."""
    text = str(text).replace("\xa0", " ")
    m = re.search(r'\(\s*단위\s*[:\uff1a]\s*([^)]+?)\s*\)', text)
    if m:
        return m.group(1).strip()
    m = re.search(r'단위\s*[:\uff1a]\s*(\S+)', text)
    if m:
        return m.group(1).strip()
    return None


# 테스트 케이스
test_cases = [
    ("1,234,567",      1234567.0),
    ("(1,234,567)",   -1234567.0),
    ("-",              0.0),
    ("–",              0.0),
    ("",               None),
    ("자산총계",         None),
    ("50,000",         50000.0),
    ("(50,000)",      -50000.0),
]

print(f"{'입력':<20} {'기대값':<15} {'결과':<15} {'통과'}")
print("-" * 60)
all_pass = True
for raw, expected in test_cases:
    result = parse_numeric(raw)
    passed = result == expected
    all_pass = all_pass and passed
    print(f"{raw!r:<20} {str(expected):<15} {str(result):<15} {'O' if passed else 'X'}")

print()
print(f"전체 테스트: {'모두 통과' if all_pass else '일부 실패'}")

입력                   기대값             결과              통과
------------------------------------------------------------
'1,234,567'          1234567.0       1234567.0       O
'(1,234,567)'        -1234567.0      -1234567.0      O
'-'                  0.0             0.0             O
'–'                  0.0             0.0             O
''                   None            0.0             X
'자산총계'               None            None            O
'50,000'             50000.0         50000.0         O
'(50,000)'           -50000.0        -50000.0        O

전체 테스트: 일부 실패


In [70]:
# [실습 6-2] DataFrame 수치 컬럼 일괄 정제

def clean_financial_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    재무제표 DataFrame의 수치 컬럼에 parse_numeric()을 적용한다.
    - 첫 번째 컬럼: 계정과목명 (텍스트 유지)
    - 나머지 컬럼: 수치 변환 시도
    - NaN 컬럼명 정리
    """
    if df is None or df.empty:
        return df

    df = df.copy()

    # NaN 컬럼명 정리
    new_cols = []
    for i, col in enumerate(df.columns):
        s = str(col).strip()
        new_cols.append(s if s not in ("nan", "", "None") else f"col_{i}")
    df.columns = new_cols

    # 수치 컬럼 변환 (첫 번째 컬럼 제외)
    for col in df.columns[1:]:
        df[col] = df[col].apply(parse_numeric)

    return df.reset_index(drop=True)


# 재무상태표 정제 적용
df_bs_raw   = raw_dfs.get("재무상태표")
df_bs_clean = clean_financial_df(df_bs_raw)

if df_bs_clean is not None:
    print(f"정제 전: {df_bs_raw.shape}  →  정제 후: {df_bs_clean.shape}")
    print()
    print(df_bs_clean.iloc[:, :4].head(15).to_string())

정제 전: (47, 6)  →  정제 후: (47, 6)

                        과 목       주석   제 56 (당) 기  제 56 (당) 기.1
0                       자 산      NaN          NaN           NaN
1                Ⅰ. 유 동 자 산      NaN          NaN    82320322.0
2               1. 현금및현금성자산    428.0    1653766.0           NaN
3                 2. 단기금융상품    428.0   10187991.0           NaN
4                   3. 매출채권  45728.0   33840357.0           NaN
5                    4. 미수금   4728.0    3249731.0           NaN
6                   5. 선급비용      NaN    1381781.0           NaN
7                   6. 재고자산      8.0   29154115.0           NaN
8                 7. 기타유동자산    428.0    2852581.0           NaN
9              Ⅱ. 비 유 동 자 산      NaN          NaN   242645805.0
10       1. 기타포괄손익-공정가치금융자산   4628.0    2176346.0           NaN
11         2. 당기손익-공정가치금융자산   4628.0          0.0           NaN
12  3. 종속기업, 관계기업 및 공동기업 투자      9.0   57427196.0           NaN
13                  4. 유형자산     10.0  151446870.0           NaN
14     

In [71]:
# [실습 6-3] 테이블 앞 문맥에서 단위 정보 탐색

def extract_unit_from_context(table_tag: Tag) -> Optional[str]:
    """
    테이블 이전 최대 10개 형제 노드에서 단위 정보를 탐색한다.
    없으면 테이블 자체 텍스트에서 탐색한다.
    """
    node = table_tag.previous_sibling
    checked = 0
    while node and checked < 10:
        if isinstance(node, Tag):
            unit = extract_unit_label(node.get_text(" ", strip=True))
            if unit:
                return unit
            checked += 1
        node = node.previous_sibling
    return extract_unit_label(table_tag.get_text(" ", strip=True))


# 5개 재무제표의 단위 정보 확인
print("[재무제표별 단위 정보]")
for stmt, tbl in found_tables.items():
    if tbl:
        unit = extract_unit_from_context(tbl)
        print(f"  {stmt:<15}: {unit or '(미확인)'}")

[재무제표별 단위 정보]
  재무상태표          : 백만원
  손익계산서          : 백만원
  포괄손익계산서        : 백만원
  자본변동표          : 백만원
  현금흐름표          : 백만원


---

## Step 7. 전체 재무제표 추출 파이프라인

### 7.1 파이프라인(Pipeline)이란

소프트웨어에서 **파이프라인**이란 데이터가 여러 처리 단계를 순서대로 통과하는 구조이다.
공장의 조립 라인과 유사하게, 앞 단계의 출력이 다음 단계의 입력이 된다.

```
[입력] HTML table 태그
    │
    ▼
[1단계] table_tag_to_df()      ← read_html() 또는 수동 파싱
    │
    ▼
[2단계] clean_financial_df()   ← 수치 변환, NaN 컬럼명 정리
    │
    ▼
[3단계] extract_unit_from_context()  ← 단위 정보 추출
    │
    ▼
[출력] .csv 파일 저장
```

앞서 구현한 모든 함수를 통합하여 5개 재무제표를 일괄 처리하고 CSV로 저장한다.

### 7.2 결과 딕셔너리 구조

각 재무제표에 대해 아래 정보를 딕셔너리로 저장한다.

```python
{
    "status"   : "ok",           # "ok" | "not_found" | "empty"
    "code"     : "BS",           # 재무제표 코드
    "csv_path" : "output/...",   # 저장된 CSV 경로
    "shape"    : (50, 4),        # DataFrame 크기 (행 수, 열 수)
    "unit"     : "백만원",        # 금액 단위
    "df"       : DataFrame,      # 정제된 DataFrame 객체
}
```

In [72]:
# [실습 7-1] 재무제표 추출 파이프라인 실행

tables_dir = OUTPUT_DIR / "processed" / "tables" / str(YEAR)
tables_dir.mkdir(parents=True, exist_ok=True)

extraction_results: Dict[str, Any] = {}

for stmt_type, code in STATEMENT_CODES.items():
    table_tag = found_tables.get(stmt_type)

    if table_tag is None:
        extraction_results[stmt_type] = {"status": "not_found"}
        continue

    df_raw   = table_tag_to_df(table_tag)
    df_clean = clean_financial_df(df_raw)
    unit     = extract_unit_from_context(table_tag)

    if df_clean is None or df_clean.empty:
        extraction_results[stmt_type] = {"status": "empty"}
        continue

    csv_name = f"{YEAR}_{code}_{stmt_type}.csv"
    csv_path = tables_dir / csv_name
    df_clean.to_csv(csv_path, index=False, encoding="utf-8-sig")

    extraction_results[stmt_type] = {
        "status"   : "ok",
        "code"     : code,
        "csv_path" : str(csv_path),
        "shape"    : df_clean.shape,
        "unit"     : unit,
        "df"       : df_clean,
    }

# 결과 요약
print(f"{'재무제표':<15} {'코드':<6} {'상태':<10} {'형태':<15} {'단위'}")
print("-" * 60)
for stmt, info in extraction_results.items():
    code  = STATEMENT_CODES.get(stmt, "")
    st    = info.get("status", "")
    shape = str(info.get("shape", ""))
    unit  = info.get("unit") or ""
    print(f"{stmt:<15} {code:<6} {st:<10} {shape:<15} {unit}")

재무제표            코드     상태         형태              단위
------------------------------------------------------------
재무상태표           BS     ok         (47, 6)         백만원
손익계산서           IS     ok         (15, 6)         백만원
포괄손익계산서         OCI    ok         (7, 6)          백만원
자본변동표           CSE    ok         (17, 7)         백만원
현금흐름표           CF     ok         (31, 6)         백만원


/var/folders/cv/gqggtkw96sx21g79rcxw1rph0000gn/T/ipykernel_25136/2237354713.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(table_tag), flavor="lxml")
/var/folders/cv/gqggtkw96sx21g79rcxw1rph0000gn/T/ipykernel_25136/2237354713.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(table_tag), flavor="lxml")
/var/folders/cv/gqggtkw96sx21g79rcxw1rph0000gn/T/ipykernel_25136/2237354713.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(table_tag), flavor="lxml")
/var/folders/cv/gqggtkw96sx21g79rcxw1rph0000gn/T/ipykernel_25136/2237354713.py:12: Futur

In [73]:
# [실습 7-2] 재무상태표 + 손익계산서 주요 항목 확인

for stmt in ["재무상태표", "손익계산서"]:
    info = extraction_results.get(stmt, {})
    if info.get("status") != "ok":
        print(f"{stmt}: 추출 실패")
        continue

    df = info["df"]
    print(f"\n{'='*60}")
    print(f"{stmt}  (단위: {info.get('unit', '미확인')})")
    print(f"크기: {df.shape}")
    print(df.iloc[:, :4].head(12).to_string())


재무상태표  (단위: 백만원)
크기: (47, 6)
                   과 목       주석  제 56 (당) 기  제 56 (당) 기.1
0                  자 산      NaN         NaN           NaN
1           Ⅰ. 유 동 자 산      NaN         NaN    82320322.0
2          1. 현금및현금성자산    428.0   1653766.0           NaN
3            2. 단기금융상품    428.0  10187991.0           NaN
4              3. 매출채권  45728.0  33840357.0           NaN
5               4. 미수금   4728.0   3249731.0           NaN
6              5. 선급비용      NaN   1381781.0           NaN
7              6. 재고자산      8.0  29154115.0           NaN
8            7. 기타유동자산    428.0   2852581.0           NaN
9         Ⅱ. 비 유 동 자 산      NaN         NaN   242645805.0
10  1. 기타포괄손익-공정가치금융자산   4628.0   2176346.0           NaN
11    2. 당기손익-공정가치금융자산   4628.0         0.0           NaN

손익계산서  (단위: 백만원)
크기: (15, 6)
                 과 목     주 석  제 56 (당) 기  제 56 (당) 기.1
0           Ⅰ. 매 출 액    29.0         NaN   209052241.0
1         Ⅱ. 매 출 원 가    21.0         NaN   152061472.0
2       Ⅲ. 매 출 총 이 익 

---

## Step 8. 주석 계층 구조 파싱

### 8.1 주석(Notes)이란 무엇인가

감사보고서의 **주석(注釋)** 섹션은 재무제표 본문에 기재된 숫자들의 상세 내용을 설명한다.
예를 들어 재무상태표의 "현금및현금성자산" 항목 옆에 "주석 4"라고 표시되어 있으면,
주석 4번에서 현금 구성 내역, 제한 사항 등이 자세히 설명된다.

주석은 매우 방대하며, 이 감사보고서에서는 문서 전체의 절반 이상을 차지한다.

---

### 8.2 주석 섹션의 5단계 계층 구조

주석은 아래와 같은 패턴으로 계층이 구성된다.

| 레벨 | 패턴 | 예시 | 의미 |
|:---:|:---|:---|:---|
| **L1** | `숫자.` 또는 `숫자)` | `1. 일반사항` | 주석 대분류 |
| **L2** | `숫자.숫자` | `2.1 재무제표 작성 기준` | L1의 세부 항목 |
| **L3** | `한글자.` | `가. 측정 기준` | 세부 소제목 |
| **L4** | `(숫자)` | `(1) 공정가치 측정` | 항목 번호 |
| **L5** | `숫자)` | `1) 당기손익 인식 항목` | 가장 세밀한 구분 |

이 패턴들은 **정규표현식(regex)**으로 판별한다.

---

### 8.3 왜 계층 구조로 파싱해야 하는가

단순히 텍스트를 평탄하게 나열하면 어떤 설명이 어느 주석 번호에 속하는지 알 수 없다.
계층 트리로 구성해야 "주석 2.1의 내용", "주석 4 아래의 표" 등을 정확히 특정할 수 있다.
이후 재무제표의 계정과목과 주석을 연결하는 메타데이터 작업에 필수적이다.

---

### 8.4 스택(Stack) 알고리즘 원리

**스택**은 "마지막에 넣은 것을 먼저 꺼내는(LIFO)" 자료구조이다.
파이썬 리스트를 스택으로 사용할 수 있다: `list.append()` = push, `list.pop()` = pop.

계층 구조를 구성하는 알고리즘:

```
스택: []                       # 초기 상태

"1. 일반사항" (L1) 처음 발견:
  스택에서 L1 이상 제거 → 없음
  루트에 추가
  스택: [(노드_1, L1)]

"1.1 회사 개요" (L2) 발견:
  스택에서 L2 이상 제거 → 없음 (L1만 있음)
  스택 최상단(노드_1)의 자식으로 추가
  스택: [(노드_1, L1), (노드_1.1, L2)]

"1.2 설립일" (L2) 발견:
  스택에서 L2 이상 제거 → (노드_1.1, L2) 제거
  스택 최상단(노드_1)의 자식으로 추가
  스택: [(노드_1, L1), (노드_1.2, L2)]

"2. 재무제표 작성 기준" (L1) 발견:
  스택에서 L1 이상 제거 → (노드_1.2, L2), (노드_1, L1) 모두 제거
  루트에 추가
  스택: [(노드_2, L1)]
```

핵심 아이디어: **레벨이 같거나 높은 노드를 스택에서 꺼내면** 자동으로 올바른 부모를 찾게 된다.

In [74]:
# [실습 8-1] 헤딩 패턴 분류 함수

HEADING_PATTERNS: List[Tuple[re.Pattern, Optional[int]]] = [
    (re.compile(r'^\s*(\d+(?:\.\d+)+)\s+(.+)$'), None),  # L2+: 2.1, 2.1.3
    (re.compile(r'^\s*(\d+)\s*[.)]\s+(.+)$'),    1),      # L1 : 2. / 2)
    (re.compile(r'^\s*([가-힣])\.\s*(.+)$'),      3),      # L3 : 가.
    (re.compile(r'^\s*\(\s*(\d+)\s*\)\s+(.+)$'),  4),      # L4 : (1)
    (re.compile(r'^\s*(\d+)\)\s+(.+)$'),          5),      # L5 : 1)
]


def classify_heading(text: str) -> Optional[Tuple[int, str, str]]:
    """
    텍스트가 헤딩 패턴과 일치하면 (레벨, 번호, 제목) 튜플을 반환한다.
    점 포함 숫자(2.1 등)는 점 개수+1을 레벨로 사용한다.
    """
    s = text.strip()
    if not s:
        return None

    for pattern, fixed_level in HEADING_PATTERNS:
        m = pattern.match(s)
        if not m:
            continue
        num   = m.group(1)
        title = m.group(2).strip()
        level = (1 + num.count(".")) if fixed_level is None else fixed_level
        return (min(level, 5), num, title)

    return None


# 헤딩 분류 테스트
test_headings = [
    "1. 일반사항",
    "2.1 재무제표 작성 기준",
    "2.1.3 수익 인식 기준",
    "가. 측정 기준",
    "(1) 공정가치 측정",
    "1) 당기손익 인식 항목",
    "일반적인 본문 텍스트입니다.",
    "당기순이익은 증가하였습니다.",
]

print(f"{'입력':<35} {'레벨':<6} {'번호':<10} {'제목'}")
print("-" * 75)
for text in test_headings:
    r = classify_heading(text)
    if r:
        level, num, title = r
        print(f"{text!r:<35} L{level:<5} {num!r:<10} {title!r}")
    else:
        print(f"{text!r:<35} (일반 본문)")

입력                                  레벨     번호         제목
---------------------------------------------------------------------------
'1. 일반사항'                           L1     '1'        '일반사항'
'2.1 재무제표 작성 기준'                    L2     '2.1'      '재무제표 작성 기준'
'2.1.3 수익 인식 기준'                    L3     '2.1.3'    '수익 인식 기준'
'가. 측정 기준'                          L3     '가'        '측정 기준'
'(1) 공정가치 측정'                       L4     '1'        '공정가치 측정'
'1) 당기손익 인식 항목'                     L1     '1'        '당기손익 인식 항목'
'일반적인 본문 텍스트입니다.'                   (일반 본문)
'당기순이익은 증가하였습니다.'                   (일반 본문)


In [75]:
# [실습 8-2] 주석 섹션 노드 수집

def collect_notes_nodes(soup_obj: BeautifulSoup) -> List[Tag]:
    """주석 섹션(id=toc_4) 이후 다음 SECTION-1 전까지의 노드를 수집한다."""
    notes_header = soup_obj.find("h3", id="toc_4")
    if notes_header is None:
        # id 없이 텍스트로 탐색
        for h3 in soup_obj.find_all("h3"):
            if "주석" in h3.get_text(strip=True):
                notes_header = h3
                break

    if notes_header is None:
        return []

    nodes: List[Tag] = []
    for sib in notes_header.next_siblings:
        if not isinstance(sib, Tag):
            continue
        if is_section_header(sib) and get_section_level(sib) == 1:
            break
        nodes.append(sib)

    return nodes


notes_nodes = collect_notes_nodes(soup)
p_nodes   = [n for n in notes_nodes if n.name == "p"]
tbl_nodes = [n for n in notes_nodes if n.name == "table"]

print(f"주석 섹션 전체 노드: {len(notes_nodes)}")
print(f"  <p>     노드: {len(p_nodes)}")
print(f"  <table> 노드: {len(tbl_nodes)}")

주석 섹션 전체 노드: 651
  <p>     노드: 408
  <table> 노드: 243


In [76]:
# [실습 8-3] 스택 알고리즘으로 계층 트리 구성

def build_notes_tree(nodes: List[Tag]) -> List[Dict[str, Any]]:
    """
    주석 섹션 노드들을 순회하여 5단계 계층 트리를 구성한다.

    스택은 (노드 딕셔너리, 레벨) 튜플의 리스트로 관리된다.
    새 헤딩이 나타나면:
    1. 자신 이상 레벨의 스택 노드를 팝한다.
    2. 새 노드를 스택 최상단의 자식(subnotes)으로 추가한다.
    3. 새 노드를 스택에 추가한다.
    """
    root: List[Dict[str, Any]] = []
    stack: List[Tuple[Dict[str, Any], int]] = []

    def new_node(num: str, title: str) -> Dict[str, Any]:
        return {"note_no": num, "note_title": title, "blocks": [], "subnotes": []}

    def attach(node: Dict, level: int) -> None:
        while stack and stack[-1][1] >= level:
            stack.pop()
        if stack:
            stack[-1][0]["subnotes"].append(node)
        else:
            root.append(node)
        stack.append((node, level))

    for tag_node in nodes:
        # 페이지 브레이크 건너뜀
        if tag_node.name == "p" and "PGBRK" in (tag_node.get("class") or []):
            continue

        if tag_node.name in ("p", "div"):
            raw = tag_node.get_text(" ", strip=True).replace("\xa0", " ")
            lines = [l.strip() for l in raw.split("\n") if l.strip()] or [raw]

            for line in lines:
                result = classify_heading(line)
                if result:
                    level, num, title = result
                    attach(new_node(num, title), level)
                elif stack and line:
                    stack[-1][0]["blocks"].append({"type": "paragraph", "text": line})

        elif tag_node.name == "table" and stack:
            stack[-1][0]["blocks"].append(
                {"type": "table", "row_count": len(tag_node.find_all("tr"))}
            )

    return root


notes_tree = build_notes_tree(notes_nodes)
print(f"L1 노드 수 (최상위 주석 항목): {len(notes_tree)}")

# L1 항목 목록 출력
print()
print("[L1 주석 항목 목록]")
for node in notes_tree:
    sub_count = len(node.get("subnotes", []))
    print(f"  [{node['note_no']}] {node['note_title'][:50]}  (하위: {sub_count})")

L1 노드 수 (최상위 주석 항목): 31

[L1 주석 항목 목록]
  [1] 일반적 사항: 삼성전자주식회사(이하 "회사")는 1969년 대한민국에서 설립되어 1975년  (하위: 0)
  [2] 중요한 회계처리방침:  (하위: 17)
  [3] 중요한 회계추정 및 가정:  (하위: 7)
  [4] 범주별 금융상품:  (하위: 2)
  [5] 금융자산의 양도: 회사는 당기 및 전기 중 은행과의 매출채권 팩토링 계약을 통해 매출채권을  (하위: 0)
  [6] 공정가치금융자산:  (하위: 4)
  [7] 매출채권 및 미수금:  (하위: 4)
  [8] 재고자산:  (하위: 0)
  [9] 종속기업, 관계기업 및 공동기업 투자:  (하위: 5)
  [10] 유형자산: 가. 당기 및 전기 중 유형자산의 변동 내역은 다음과 같습니다. (1) 당기  (하위: 3)
  [11] 무형자산: 가. 당기 및 전기 중 무형자산의 변동 내역은 다음과 같습니다.  (하위: 3)
  [12] 차입금:  (하위: 3)
  [13] 사채:  (하위: 2)
  [14] 순확정급여부채(자산): 가. 보고기간종료일 현재 순확정급여부채(자산)의 내역은 다음과 같습  (하위: 8)
  [15] 충당부채: 당기 중 충당부채의 변동 내역은 다음과 같습니다.  (하위: 5)
  [16] 우발부채와 약정사항:  (하위: 4)
  [17] 계약부채: 보고기간종료일 현재 계약부채는 다음과 같습니다.  (하위: 0)
  [18] 자본금:  (하위: 0)
  [19] 이익잉여금:  (하위: 3)
  [20] 기타자본항목: 가. 보고기간종료일 현재 기타자본항목의 내역은 다음과 같습니다.  (하위: 1)
  [21] 비용의 성격별 분류:  (하위: 0)
  [22] 판매비와관리비: 당기 및 전기 중 판매비와관리비의 내역은 다음과 같습니다.  (하위: 0)
  [23] 기타수익 및 기타비용:  (하위: 0)
  [24] 금융수익 및 금융비용: 당기 및 전기 중 금융수익 및 금융비용의 내역은 다음과 같습니다.  (

In [77]:
# [실습 8-4] 트리 구조 미리보기 (2단계까지)

def preview_tree(
    nodes: List[Dict],
    indent: int = 0,
    max_items: int = 3,
    max_depth: int = 2,
) -> None:
    """트리를 들여쓰기 형태로 출력한다."""
    if indent >= max_depth * 2:
        return
    for i, node in enumerate(nodes):
        if i >= max_items:
            print(" " * indent + f"  ... ({len(nodes) - max_items}개 더)")
            break
        b = len(node.get("blocks", []))
        s = len(node.get("subnotes", []))
        print(" " * indent + f"[{node['note_no']}] {node['note_title'][:45]}")
        if node.get("subnotes"):
            preview_tree(node["subnotes"], indent + 2, max_items, max_depth)


print("[주석 트리 (2단계 미리보기)]")
preview_tree(notes_tree, max_items=4, max_depth=2)

[주석 트리 (2단계 미리보기)]
[1] 일반적 사항: 삼성전자주식회사(이하 "회사")는 1969년 대한민국에서 설립되어 
[2] 중요한 회계처리방침:
  [2.1] 재무제표 작성기준
  [2.2] 회계정책과 공시의 변경
  [2.3] 종속기업, 관계기업 및 공동기업
  [2.4] 외화환산
    ... (13개 더)
[3] 중요한 회계추정 및 가정:
  [가] 수익인식
  [나] 판매보증충당부채
  [다] 금융상품의 공정가치
  [라] 금융자산의 손상 회사는 금융자산의 손실충당금을 측정할 때에 채무불이행위험과 기대신
    ... (3개 더)
[4] 범주별 금융상품:
  [가] 보고기간종료일 현재 범주별 금융상품 내역은 다음과 같습니다. (1) 당기말
  [나] 금융상품 범주별 순손익 구분 (1) 당기
  ... (27개 더)


---

## Step 9. 결과 저장

### 9.1 저장 형식 선택 기준

파이프라인에서 추출한 데이터는 두 가지 형식으로 저장한다.

| 형식 | 특징 | 저장 데이터 |
|:---|:---|:---|
| **JSON** | 계층 구조(트리)를 그대로 표현 가능. 사람이 읽기 쉬움 | 주석 계층 트리 |
| **CSV** | 스프레드시트·DB에 바로 로드 가능. 2차원 표 구조 | 재무제표 DataFrame, 메타데이터 |

**JSON(JavaScript Object Notation)** 은 중괄호 `{}` 와 대괄호 `[]` 로 중첩 구조를 표현하는 텍스트 포맷이다.
```json
{
  "note_no": "1",
  "note_title": "일반사항",
  "subnotes": [
    { "note_no": "1.1", "note_title": "회사 개요" }
  ]
}
```

**CSV(Comma-Separated Values)** 는 쉼표로 열을 구분하는 단순 텍스트 표 형식이다.
Excel, pandas, SQL 등 모든 데이터 도구에서 지원한다.

---

### 9.2 출력 디렉토리 구조

```
output/
  ├── processed/
  │   ├── jsons/
  │   │   └── 2024/
  │   │       ├── sections.json    ← 2회차 출력: 섹션 텍스트
  │   │       └── notes.json       ← 3회차 출력: 주석 트리
  │   └── tables/
  │       └── 2024/
  │           ├── 2024_BS_재무상태표.csv
  │           ├── 2024_IS_손익계산서.csv
  │           ├── 2024_OCI_포괄손익계산서.csv
  │           ├── 2024_CSE_자본변동표.csv
  │           └── 2024_CF_현금흐름표.csv
  └── structured/
      └── 2024/
          └── 2024_meta.csv        ← 재무제표-주석 연결 메타데이터
```

In [78]:
# [실습 9-1] 주석 트리 → JSON 저장

json_out_dir = OUTPUT_DIR / "processed" / "jsons" / str(YEAR)
json_out_dir.mkdir(parents=True, exist_ok=True)

def make_serializable(node: Dict) -> Dict:
    """Tag 객체 등 JSON 직렬화 불가 항목을 제거한다."""
    return {
        "note_no"    : node.get("note_no", ""),
        "note_title" : node.get("note_title", ""),
        "blocks"     : [
            {k: v for k, v in b.items() if isinstance(v, (str, int, float, bool, type(None)))}
            for b in node.get("blocks", [])
        ],
        "subnotes"   : [make_serializable(c) for c in node.get("subnotes", [])],
    }


notes_payload = {
    "report_id"  : YEAR,
    "source_file": HTML_PATH.name,
    "notes"      : [make_serializable(n) for n in notes_tree],
}

notes_json_path = json_out_dir / "notes.json"
with notes_json_path.open("w", encoding="utf-8") as f:
    json.dump(notes_payload, f, ensure_ascii=False, indent=2)

print(f"주석 JSON 저장: {notes_json_path}")
print(f"파일 크기: {notes_json_path.stat().st_size:,} bytes")

주석 JSON 저장: ../output/processed/jsons/2024/notes.json
파일 크기: 102,676 bytes


In [79]:
# [실습 9-2] 재무제표-주석 연결 메타데이터 저장

meta_rows: List[Dict[str, Any]] = []

for stmt_type, info in extraction_results.items():
    if info.get("status") != "ok":
        continue

    df   = info["df"]
    code = STATEMENT_CODES.get(stmt_type, "")
    unit = info.get("unit") or ""

    account_col = df.columns[0]
    # 두 번째 컬럼에 주석 번호가 있다고 가정
    note_col = df.columns[1] if len(df.columns) > 1 else None

    for _, row in df.iterrows():
        account = str(row[account_col]).strip()
        if not account or account == "nan":
            continue

        note_ref = ""
        if note_col:
            raw_note = row.get(note_col)
            if raw_note and str(raw_note) not in ("nan", "None", ""):
                nums = re.findall(r'\d+', str(raw_note))
                note_ref = ",".join(nums)

        meta_rows.append({
            "report_id"     : YEAR,
            "statement_type": code,
            "account_name"  : account,
            "note_ref"      : note_ref,
            "unit"          : unit,
        })

meta_df = pd.DataFrame(meta_rows)

struct_dir = OUTPUT_DIR / "structured" / str(YEAR)
struct_dir.mkdir(parents=True, exist_ok=True)
meta_csv_path = struct_dir / f"{YEAR}_meta.csv"
meta_df.to_csv(meta_csv_path, index=False, encoding="utf-8-sig")

print(f"메타데이터 저장: {meta_csv_path}")
print(f"총 {len(meta_df)}행")
print()
# 주석 참조가 있는 행만 표시
with_note = meta_df[meta_df["note_ref"] != ""]
print(f"주석 참조 있는 항목: {len(with_note)}개")
print(with_note.head(8).to_string(index=False))

메타데이터 저장: ../output/structured/2024/2024_meta.csv
총 117행

주석 참조 있는 항목: 54개
 report_id statement_type       account_name note_ref unit
      2024             BS        1. 현금및현금성자산    428,0  백만원
      2024             BS          2. 단기금융상품    428,0  백만원
      2024             BS            3. 매출채권  45728,0  백만원
      2024             BS             4. 미수금   4728,0  백만원
      2024             BS            6. 재고자산      8,0  백만원
      2024             BS          7. 기타유동자산    428,0  백만원
      2024             BS 1. 기타포괄손익-공정가치금융자산   4628,0  백만원
      2024             BS   2. 당기손익-공정가치금융자산   4628,0  백만원


In [80]:
# [실습 9-3] 저장된 전체 파일 목록 확인

print("[저장된 출력 파일 목록]")
print(f"{'파일명':<55} {'크기(KB)':>8}")
print("-" * 66)
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        rel  = f.relative_to(OUTPUT_DIR)
        size = f.stat().st_size / 1024
        print(f"{str(rel):<55} {size:>8.1f}")

[저장된 출력 파일 목록]
파일명                                                       크기(KB)
------------------------------------------------------------------
processed/jsons/2024/notes.json                            100.3
processed/jsons/2024/sections.json                         119.1
processed/tables/2024/2024_BS_재무상태표.csv                      2.2
processed/tables/2024/2024_CF_현금흐름표.csv                      1.9
processed/tables/2024/2024_CSE_자본변동표.csv                     1.2
processed/tables/2024/2024_IS_손익계산서.csv                      0.8
processed/tables/2024/2024_OCI_포괄손익계산서.csv                   0.5
structured/2024/2024_meta.csv                                5.9


---

## 핵심 개념 정리

| 개념 | 설명 |
|:---|:---|
| `rowspan` / `colspan` | HTML 셀 병합 속성. 재무제표 헤더에 빈번히 사용됨 |
| `pd.read_html()` | HTML 문자열에서 테이블을 DataFrame으로 자동 변환 |
| 괄호 음수 표현 | `(1,234)` → `-1234.0`. 한국 재무제표 표준 표기 |
| 키워드 검증 | 테이블 내 고유 계정과목 키워드 출현으로 재무제표 유형 판별 |
| 스택 기반 트리 구성 | 헤딩 레벨이 올라가면 스택을 팝하여 계층 관계를 유지하는 알고리즘 |
| 방어적 폴백(fallback) | `read_html()` 실패 시 수동 파싱으로 자동 대체하는 설계 패턴 |

---

## 다음 단계: 데이터 활용

본 실습에서 구축한 파이프라인으로 이후 과정에서 다음 분석을 수행할 수 있다.

- **다년도 재무 트렌드 분석**: 2015~2024년 데이터를 연도별로 추출하여 시계열 분석
- **재무비율 자동 계산**: 유동비율, 부채비율, ROE, ROA 등
- **주석 텍스트 분석**: 회계정책 변경 탐지, 비교 분석
- **LLM 연계**: 구조화 데이터와 비구조화 주석 텍스트를 LLM 입력으로 활용